<a href="https://colab.research.google.com/github/jkrescue/nvidia-physicsnemo-use/blob/master/PhysicsNemoUsage01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 使用PhysicsNeMo
physicsnemo相比于以往的moduls或者physicsnemo-CFD等，如下热点：

👉 PhysicsNeMo = 新一代统一框架（正在整合）

👉 PhysicsNeMo-Sym（Symbolic）= 旧的 PINN / PDE symbolic 系统（核心但逐步被吸收）

👉 physicsnemo-CFD = PhysicsNeMo 的一个“CFD推理与工程集成子模块（inference + workflow层）

```
     PhysicsNeMo（现在主线）
                 │
     ┌───────────┼───────────┐
     │                       │
 Symbolic（旧核心）      CFD / Earth-2
(PINN / PDE DSL)        (工程子集)
```

## PhysicsNeMo
NVIDIA 推的新一代 AI4Science 框架
- **✅ 本质**
> 一个“**统一容器**”，目标是整合：
> - `PINN（Symbolic）`
> - `Neural Operator（FNO / Fourier）`
> - `CFD surrogate`
> - 多物理场

- **✅ 特点**
> - `Hydra config`
> - 多模型统一接口
> - 更偏“平台化”

- **✅ 支持**
> - `FNO`
> - `PINN`（部分接入）

## PhysicsNeMo Symbolic


  👉 这个才是“老牌核心”，以前叫`NVIDIA Modulus`后改名为`PhysicsNeMo-Sym`。

- **✅ 核心能力**
> - `PDE symbolic` 表达
> - `PINN`
> - 自动微分 PDE (AD)
> - `boundary / constraint system`

- **❗ 现状**

  正在被 PhysicsNeMo 吸收，但目前repo 分裂，pip 包混乱，有时装不到。如在PhysicsNeMo/examples中执行如pinns的demo会提示：`ModuleNotFoundError: physicsnemo.sym`，所以还需要用户安装physicsnemo.sym这部分。



## PhysicsNeMo-CFD
  **“PhysicsNeMo-CFD 是 PhysicsNeMo 的一个子模块，用于把训练好的 AI 模型集成进 CFD 工程流程。”** 是 PhysicsNeMo 的一个子模块，用于把训练好的 AI 模型集成进 CFD 工程流程,不是：❌ 训练框架, ❌ Symbolic（PDE）, ❌ 简单 example。这一点不同于`PhysicsNeMo/examples/cfd`。


- **1️⃣ 推理（Inference）——核心能力**

  ```python
  from physicsnemo.cfd.inference.domino_nim import call_domino_nim
  ```

  👉 它做的是：
  * 调用 **预训练模型（NIM服务）**
  * 输入：几何（STL）、参数
  * 输出：流场预测


  👉 本质：

  ```text
  传统 CFD（OpenFOAM）
      ↓
  AI surrogate（PhysicsNeMo训练）
      ↓
  👉 physicsnemo-cfd 把它接进工程系统
  ```

---

- **2️⃣ Benchmark（评估体系）**

  提供：

  * pointwise error
  * PDE residual
  * spectral metrics
  * integrated quantities

  👉 用来做：**AI CFD vs 传统 CFD 的对比验证**

---

- **3️⃣ Hybrid CFD（工业很关键）**

  👉 很重要：**用 AI 初始化 CFD，加速收敛**

  **流程：**

  ```text
  AI 模型预测初值
          ↓
  作为 CFD solver 初始条件
          ↓
  减少迭代时间
  ```

  👉 这是工业落地的关键场景

---

> **它和 physicsnemo（主 repo）的关系**

  ```text
  [训练层]
  PhysicsNeMo（主框架）
    ├── FNO / GNN / PINN
    └── 训练 pipeline

  [部署 / 推理层]
  physicsnemo-cfd
    ├── inference API
    ├── benchmark
    └── CFD workflow 集成
  ```

---


> **关键理解**

**👉 physicsnemo-cfd ≠ 做 CFD**

  👉 不做：

  * Navier-Stokes 求解
  * PDE 建模
  * PINN

而是：**“把 AI 模型嵌入 CFD 工程体系”**。

这点和传统 CFD 框架（OpenFOAM）完全不同：

| 系统              | 本质           |
| --------------- | ------------ |
| OpenFOAM        | 数值求解器        |
| PhysicsNeMo     | AI训练框架       |
| physicsnemo-cfd | AI → CFD 接口层 |

---

> **为什么 单独搞这个 repo？**

因为真实工业需求是：

```text
不是训练模型 ❌
而是用模型 ✔
```

---

## 工业流程是：

```text
1️⃣ 用 PhysicsNeMo 训练模型
2️⃣ 部署成 NIM（推理服务）
3️⃣ 用 physicsnemo-cfd 调用 + 集成
4️⃣ 接入 CFD workflow（OpenFOAM / CAE）
```
这个 repo 就在做：**第 3 步 + 第 4 步**

---


In [2]:
# 由于当前环境是colab，所以使用 google drive 以便保存当前记录，需要mount google drive，如下：

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
# 定位到当前workspace，之后工作会在这个workspace展开。
%cd /content/drive/MyDrive/AIforScience/PhysicsNemo/

/content/drive/MyDrive/AIforScience/PhysicsNemo


In [25]:
# clone physicsnemo repo到当前workspace中，若之前clone过也可以rm -rf xxx，之后重新git clone
!git clone https://github.com/NVIDIA/physicsnemo.git

Cloning into 'physicsnemo'...
remote: Enumerating objects: 20620, done.
remote: Counting objects: 100% (695/695), done.
remote: Compressing objects: 100% (490/490), done.
remote: Total 20620 (delta 421), reused 224 (delta 181), pack-reused 19925 (from 3)
Receiving objects: 100% (20620/20620), 258.80 MiB | 17.53 MiB/s, done.
Resolving deltas: 100% (13247/13247), done.
Updating files: 100% (2362/2362), done.


In [5]:
# 定位到physicsnemo路径下，应为路径中包含了 pyproject.toml文件，可以在该路径下pip install .
# 但是若pip install physicsnemo,通过包名来安装，则pip直接从python包索引（pypi)中下载physicsnemo。

%cd /content/drive/MyDrive/AIforScience/PhysicsNemo/physicsnemo

# 检查当前路径是否包含安装项目配置文件pyproject.toml（eg setup.py, setup.cfg等）
import os
assert os.path.exists("pyproject.toml"), "当前目录不是项目根目录"
assert os.path.isdir("physicsnemo"), "没找到 physicsnemo 包目录"


/content/drive/MyDrive/AIforScience/PhysicsNemo/physicsnemo


In [6]:
# 安装physicsnemo，若之前安装过旧版本，可以卸载后重装
# !pip uninstall -y nvidia-physicsnemo physicsnemo || true
!python -m pip install --upgrade pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [8]:
# 安装nvidia-physicnemo，直接使用pypi包安装，由于使用colab，安装完成后需要重启运行环境（主要是因为之前安装过，有环境变量被写入，用户如果之前安装的正确，则不需要重新，即不会存在环境变量被写入生效的问题）
# 重启colab 运行时(如果之前没有安装过，不需要重启），注意需要重新navigate到target directory

!pip install nvidia-physicsnemo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 52.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 105.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 MB 69.7 MB/s  0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [nvidia-physicsnemo]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.


In [1]:
# 验证physicsnemo安装成功
!python -c "import physicsnemo; print('PhysicsNeMo version:', physicsnemo.__version__)"

PhysicsNeMo version: 2.0.0


In [2]:
import physicsnemo
print("physicsnemo path:", physicsnemo.__file__)

physicsnemo path: /usr/local/lib/python3.12/dist-packages/physicsnemo/__init__.py


In [3]:
import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch version: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


In [12]:
# optional, 如果需要运行包含physicsnemo.sym.xx时需要安装PhysicsNeMo Symbolic，安装PhysicsNeMo Symbolic前确保physicsnemo正常安装
!pip install "Cython"
!pip install nvidia-physicsnemo.sym --no-build-isolation

# 重启colab 运行时(如果之前没有安装过，不需要重启），注意需要重新navigate到target directory
# 当前physicsnemo已经正常安装,下一步需要针对examples安装对应的依赖
# ！！！当前 physicsnemo.sym 没有，所以cfd中用到 physicsnemo.sym.xxx的都不行。
from physicsnemo.sym.eq.pdes.navier_stokes import NavierStokes
from physicsnemo.sym.eq.phy_informer import PhysicsInformer
from physicsnemo.sym.geometry.geometry_dataloader import GeometryDatapipe
from physicsnemo.sym.geometry.primitives_2d import Rectangle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 17.4 MB/s  0:00:00
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 139.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 149.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 57.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 92.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 MB 66.2 MB/s  0:00:02
  Created wheel for nvidia-physicsnemo.sym: filename=nvidia_physicsnemo_sym-2.4.0-cp312-cp312-linux_x86_64.whl size=645518 sha256=1018be74c6686ab1b5ce030afa47f52562ccb391d548f1d34e0b33d6d2e15981
  Stored in directory: /root/.cache/pip/wheels/d6/ba/e6/a2a26601901ac7084f40e60e0889eb9d3167ec8abbc7c41394
  Created wheel for n

ModuleNotFoundError: No module named 'physicsnemo.sym'

In [4]:
# 验证physicsnemo-sym 安装成功
!python -c "import physicsnemo.sym; print('PhysicsNeMo Symbolic version:', physicsnemo.sym.__version__)"

PhysicsNeMo Symbolic version: 2.4.0


In [ ]:
# 若要执行ldc_pinns 案例，直接跳转到 执行CFD中涉及到physicsnemo.sym的部分

> **注意**：若要执行ldc_pinns 案例，请向下转到 `执行CFD中涉及到physicsnemo.sym`的部分。接下来的例子部分不涉及physicsnemo.sym。

In [1]:
# 定位到 examples 路径，接下来会执行里面的训练代码，注意: 当前使用hydra，通过config.yaml来配置训练参数，所以用户修改模型时，可以修改config.yaml或者直接通过终端参数覆盖的形式：
# python train.py \
#   training.batch_size=16 \
#   scheduler.initial_lr=5e-4 \
#   arch.fno.fno_layers=6

%cd /content/drive/MyDrive/AIforScience/PhysicsNemo/physicsnemo/examples/cfd/darcy_fno


[Errno 2] No such file or directory: '/content/drive/MyDrive/AIforScience/PhysicsNemo/physicsnemo/examples/cfd/darcy_fno'
/content


In [5]:
# 安装当前demo所需要的依赖，目前physicnemo的example中不同的demo 需要的依赖可能不同，需要用在执行的时候注意当前demo 路径中是否存在requirements.txt
!pip install -r requirements.txt

In [6]:
# 执行训练，当前没有任何针对GPU以及分布式的配置，默认选择当前的GPU
!python train_fno_darcy.py

Warp DeprecationWarning: The symbol `warp.context.Device` will soon be removed from the public API. Use `warp.Device` instead.
/usr/local/lib/python3.12/dist-packages/physicsnemo/distributed/manager.py:415: UserWarning: Could not initialize using ENV, SLURM or OPENMPI methods. Assuming this is a single process job
  warn(
[2026-03-20 05:40:30,669][checkpoint][WARNING] - Provided checkpoint directory ./checkpoints does not exist, skipping load
[2026-03-20 05:40:30,670][darcy_fno][WARNING] - Model FNO does not support AMP on GPUs, turning off
/usr/local/lib/python3.12/dist-packages/physicsnemo/utils/capture.py:274: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
[2026-03-20 05:40:30,673][darcy_fno][WARNING] - Model FNO does not support AMP on GPUs, turning off
[2026-03-20 05:40:30,674][darcy_fno][INFO] - Training started...
[2026-03-20 05:45:08,245][

In [9]:
!pip install wandb
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: buaazhangyanbo (jkzhang) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [12]:
# use wandb for visualizing results.
!python train_fno_darcy_wandb_v3.py

Warp DeprecationWarning: The symbol `warp.context.Device` will soon be removed from the public API. Use `warp.Device` instead.
/usr/local/lib/python3.12/dist-packages/physicsnemo/distributed/manager.py:415: UserWarning: Could not initialize using ENV, SLURM or OPENMPI methods. Assuming this is a single process job
  warn(
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: buaazhangyanbo (jkzhang) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run jjzramox (0.5s)
wandb: ⣯ setting up run jjzramox (0.5s)
wandb: ⣟ setting up run jjzramox (0.5s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/drive/MyDrive/AIforScience/PhysicsNemo/physicsnemo/examples/cfd/darcy_fno/wandb/run-20260320_062730-jjzra

In [13]:
# 修改配置，并执行新的训练，自动化实现, 即修改yaml中training下的batch_size的参数。
!python train_fno_darcy_wandb_v3.py training.batch_size=16

Warp DeprecationWarning: The symbol `warp.context.Device` will soon be removed from the public API. Use `warp.Device` instead.
/usr/local/lib/python3.12/dist-packages/physicsnemo/distributed/manager.py:415: UserWarning: Could not initialize using ENV, SLURM or OPENMPI methods. Assuming this is a single process job
  warn(
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: buaazhangyanbo (jkzhang) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/drive/MyDrive/AIforScience/PhysicsNemo/physicsnemo/examples/cfd/darcy_fno/wandb/run-20260320_064156-96mtdub0
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run train_fno_darcy
wandb: ⭐️ View project at https:/

# 执行CFD中涉及到physicsnemo.sym的部分

physicsnemo.sym代表的是符号计算中的部分，如用到了N-S方程或者其他符号计算中的模块，需要https://github.com/NVIDIA/physicsnemo-sym.git  这个repo的内容。

In [5]:
# 定位到 examples 路径，接下来会执行里面的训练代码，注意: 当前使用hydra，通过config.yaml来配置训练参数，所以用户修改模型时，可以修改config.yaml或者直接通过终端参数覆盖的形式：
# python train.py \
#   training.batch_size=16 \
#   scheduler.initial_lr=5e-4 \
#   arch.fno.fno_layers=6

%cd /content/drive/MyDrive/AIforScience/PhysicsNemo/physicsnemo/examples/cfd/ldc_pinns/


/content/drive/MyDrive/AIforScience/PhysicsNemo/physicsnemo/examples/cfd/ldc_pinns


In [7]:
# 若直接执行python train.py可能会提示缺少 dali包，需要安装缺失的 nvidia.dali 模块，安装完成之后重启runtime，不会影响已经安装的包，定位到目标路径，开始训练即可。
!pip install nvidia-dali-cuda120 # 或者根据你的CUDA版本选择合适的包，例如 nvidia-dali-cuda110

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.3/418.3 MB 21.2 MB/s  0:00:15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 46.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 50.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.7 MB/s  0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [nvidia-dali-cuda120]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.


In [6]:
!python train.py

Warp DeprecationWarning: The symbol `warp.context.Device` will soon be removed from the public API. Use `warp.Device` instead.
/usr/local/lib/python3.12/dist-packages/physicsnemo/distributed/manager.py:415: UserWarning: Could not initialize using ENV, SLURM or OPENMPI methods. Assuming this is a single process job
  warn(
/usr/local/lib/python3.12/dist-packages/nvidia/dali/plugin/base_iterator.py:208: Warning: Please set `reader_name` and don't set last_batch_padded and size manually whenever possible. This may lead, in some situations, to missing some samples or returning duplicated ones. Check the Sharding section of the documentation for more details.
  _iterator_deprecation_warning()
Warp DeprecationWarning: The symbol `warp.context.Device` will soon be removed from the public API. Use `warp.Device` instead.
Warp DeprecationWarning: The symbol `warp.context.Device` will soon be removed from the public API. Use `warp.Device` instead.
Loss: 0.4978722631931305, LR: 0.00099998717675862